# Pure-water evaporation inside tubes (v0.6.3)

Executed public `Simulation` and `Rating` examples for liquid preheating, partial/complete evaporation and vapor superheating. Shah (1982) heat flux is referenced to tube inside area. `zone_alpha_evaporation` is the boiling-zone coefficient; top-level `inside_alpha_equivalent` reconstructs the complete exchanger resistance.

In [1]:
from pathlib import Path
from time import perf_counter
import math
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / 'core').is_dir()), None)
if root is None:
    raise RuntimeError('Repository root was not found.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from core.geometry.bundle import TubeBundle
from core.geometry.tube import BareTube, TubeOrientation
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.heat_balance import BalanceSideSpec
from core.models.simulation import HXSideInput
from core.properties.common import FluidTransportProperties
from core.properties.fluids import ConstantPropertyProvider
from core.properties.water import IAPWS97WaterSteamProvider

P_WATER = 1.0e6
P_HOT = 101325.0
water = IAPWS97WaterSteamProvider()
hot = ConstantPropertyProvider(
    FluidTransportProperties(rho=1.2, mu=1.8e-5, k=0.026, cp=1005.0)
)

def exchanger(n_rows=10, n_tubes_per_row=10):
    return BareTubeHeatExchanger(TubeBundle(
        tube=BareTube(D_i=0.020, D_o=0.024, length_total=4.0,
                      length_effective=4.0, wall_k=16.0,
                      tube_orientation=TubeOrientation.VERTICAL_UPWARD),
        n_rows=n_rows, n_tubes_per_row=n_tubes_per_row,
        pitch_transverse=0.040, pitch_longitudinal=0.040,
        layout='inline', n_passes_tube=1, flow_arrangement='crossflow',
    ))

## Equivalent inlet state specifications

The IAPWS path accepts one of `T+p`, `p+x` or `p+h`. On the saturation line use quality or enthalpy, because `T+p` alone cannot determine vapor quality.

In [2]:
state_Tp = HXSideInput(provider=water, m_dot=1.0, T_in=350.0, p=P_WATER)
state_px = HXSideInput(provider=water, m_dot=1.0, quality_in=0.2, p=P_WATER)
state_ph = HXSideInput(provider=water, m_dot=1.0, h_in=state_px.h_in, p=P_WATER)
display(pd.DataFrame([
    {'specification': s.state_specification, 'T_K': s.T_in, 'p_Pa': s.p,
     'h_J_kg': s.h_in, 'quality': s.quality_in,
     'phase': s.water_steam_state.phase.value}
    for s in (state_Tp, state_px, state_ph)
]))
assert state_px.h_in == state_ph.h_in
assert math.isclose(state_px.quality_in, state_ph.quality_in, rel_tol=0.0, abs_tol=1e-12)

,specification,T_K,p_Pa,h_J_kg,quality,phase
0,T+p,350.000000,1000000.0,3.225012e+05,NaN,subcooled_liquid
1,p+x,453.035632,1000000.0,1.165570e+06,0.2,two_phase
2,p+h,453.035632,1000000.0,1.165570e+06,0.2,two_phase


## Simulation: partial evaporation and evaporation plus superheat

Geometry determines the available outer area. The accepted p-h outlet and area split change with bundle size.

In [3]:
outside_sim = HXSideInput(provider=hot, m_dot=30.0, T_in=700.0, p=P_HOT)
simulation_cases = {}
for name, rows in [('Simulation partial', 10), ('Simulation superheat', 25)]:
    started = perf_counter()
    result = exchanger(rows).simulate(state_Tp, outside_sim)
    simulation_cases[name] = (result, perf_counter() - started)

assert simulation_cases['Simulation partial'][0].inside_phase_change.quality_out is not None
assert simulation_cases['Simulation superheat'][0].inside_phase_change.Q_superheat > 0.0

## Rating: partial, complete, superheated and duty-controlled outlets

Rating uses the same zone physics and reports required area/UA plus actual-geometry margins. The duty-controlled case resolves `h_out = h_in + Q/m_dot` and is not clamped at `x=1`.

In [4]:
outside_rating = BalanceSideSpec(provider=hot, p=P_HOT, m_dot=30.0, T_in=700.0)
rating_specs = {
    'Rating partial': (BalanceSideSpec(provider=water, p=P_WATER, m_dot=1.0,
                                       quality_in=0.0, quality_out=0.5), None),
    'Rating complete': (BalanceSideSpec(provider=water, p=P_WATER, m_dot=1.0,
                                        T_in=350.0, quality_out=1.0), None),
    'Rating superheat': (BalanceSideSpec(provider=water, p=P_WATER, m_dot=1.0,
                                         T_in=350.0, T_out=520.0), None),
}
target_065 = water.state(x=0.65, p=P_WATER)
duty_065 = target_065.h - state_Tp.h_in
rating_specs['Rating duty x=0.65'] = (
    BalanceSideSpec(provider=water, p=P_WATER, m_dot=1.0, T_in=350.0), duty_065
)
rating_cases = {}
for name, (inside, duty) in rating_specs.items():
    started = perf_counter()
    result = exchanger().rate(inside, outside_rating, Q=duty)
    rating_cases[name] = (result, perf_counter() - started)

assert rating_cases['Rating complete'][0].inside_phase_change.quality_out == 1.0
assert rating_cases['Rating superheat'][0].inside_phase_change.Q_superheat > 0.0
assert abs(rating_cases['Rating duty x=0.65'][0].inside_phase_change.quality_out - 0.65) < 1e-12

## Public results, balances and performance diagnostics

In [5]:
def summary(name, result, runtime_s, mode):
    w = result.inside_phase_change
    return {
        'case': name, 'mode': mode, 'Q_W': w.Q_total,
        'phase_out': w.phase_out.value, 'quality_out': w.quality_out,
        'T_out_K': w.T_out, 'h_out_J_kg': w.h_out,
        'Q_preheat_W': w.Q_preheat, 'Q_evaporation_W': w.Q_evaporation,
        'Q_superheat_W': w.Q_superheat, 'A_preheat_m2': w.A_preheat,
        'A_evaporation_m2': w.A_evaporation, 'A_superheat_m2': w.A_superheat,
        'fraction_preheat': w.zone_fraction_preheat,
        'fraction_evaporation': w.zone_fraction_evaporation,
        'fraction_superheat': w.zone_fraction_superheat,
        'zone_alpha_preheat': w.zone_alpha_preheat,
        'zone_alpha_evaporation': w.zone_alpha_evaporation,
        'zone_alpha_superheat': w.zone_alpha_superheat,
        'zone_U_preheat': w.zone_U_preheat, 'zone_U_evaporation': w.zone_U_evaporation,
        'zone_U_superheat': w.zone_U_superheat, 'zone_UA_preheat': w.zone_UA_preheat,
        'zone_UA_evaporation': w.zone_UA_evaporation,
        'zone_UA_superheat': w.zone_UA_superheat, 'UA_total_W_K': w.UA_total,
        'U_equivalent_W_m2K': result.U_mean,
        'inside_alpha_equivalent': w.inside_alpha_equivalent,
        'inside_alpha_area_weighted': w.inside_alpha_area_weighted,
        'A_required_m2': getattr(result, 'A_required', None),
        'UA_required_W_K': getattr(result, 'UA_required', None),
        'UA_actual_W_K': getattr(result, 'UA_actual', None),
        'overdesign': getattr(result, 'overdesign_factor', None),
        'dp_status': w.two_phase_pressure_drop_status,
        'outer_iterations': w.iterations, 'root_iterations': w.root_iterations,
        'heat_flux_iterations': w.heat_flux_iterations,
        'property_evaluations': w.property_evaluations, 'cache_hits': w.cache_hits,
        'runtime_s': runtime_s,
        'warnings': ', '.join(sorted({warning.code for warning in w.warnings})),
    }

rows = [summary(name, result, elapsed, 'Simulation')
        for name, (result, elapsed) in simulation_cases.items()]
rows += [summary(name, result, elapsed, 'Rating')
         for name, (result, elapsed) in rating_cases.items()]
display(pd.DataFrame(rows).set_index('case'))

,mode,Q_W,phase_out,quality_out,T_out_K,h_out_J_kg,Q_preheat_W,Q_evaporation_W,Q_superheat_W,A_preheat_m2,...,UA_actual_W_K,overdesign,dp_status,outer_iterations,root_iterations,heat_flux_iterations,property_evaluations,cache_hits,runtime_s,warnings
case,,,,,,,,,,,,,,,,,,,,,
Simulation partial,Simulation,1.386023e+06,two_phase,0.469531,453.035632,1.708524e+06,440181.618579,9.458410e+05,0.000000,12.553998,...,NaN,0.000000,not_supported,25,25,26,59,155,0.362613,"WATER_BOILING_DRYOUT_CHF_NOT_MODELLED, WATER_B..."
Simulation superheat,Simulation,2.581816e+06,superheated_vapor,NaN,505.741369,2.904317e+06,440181.618579,2.014437e+06,127197.329098,23.140504,...,NaN,0.000000,not_supported,27,27,26,87,215,0.410953,"WATER_BOILING_DRYOUT_CHF_NOT_MODELLED, WATER_B..."
Rating partial,Rating,1.007218e+06,two_phase,0.500000,453.035632,1.769901e+06,0.000000,1.007218e+06,0.000000,0.000000,...,7241.458109,0.655475,not_supported,1,0,22,4,3,0.020983,"WATER_BOILING_DRYOUT_CHF_NOT_MODELLED, WATER_B..."
Rating complete,Rating,2.454618e+06,saturated_vapor,1.000000,453.035632,2.777120e+06,440181.618579,2.014437e+06,0.000000,13.417049,...,6388.193922,-0.443257,not_supported,1,0,25,6,4,0.019006,"WATER_BOILING_DRYOUT_CHF_NOT_MODELLED, WATER_B..."
Rating superheat,Rating,2.613742e+06,superheated_vapor,NaN,520.000000,2.936244e+06,440181.618579,2.014437e+06,159124.117387,13.555822,...,5913.911582,-0.528912,not_supported,1,0,25,8,5,0.024720,"WATER_BOILING_DRYOUT_CHF_NOT_MODELLED, WATER_B..."
Rating duty x=0.65,Rating,1.749565e+06,two_phase,0.650000,453.035632,2.072067e+06,440181.618579,1.309384e+06,0.000000,12.834873,...,6087.466123,-0.203341,not_supported,1,0,25,6,4,0.022695,"WATER_BOILING_DRYOUT_CHF_NOT_MODELLED, WATER_B..."


In [6]:
for name, (result, _) in {**simulation_cases, **rating_cases}.items():
    w = result.inside_phase_change
    assert math.isclose(w.Q_total, w.mass_flow_total * (w.h_out - w.h_in), rel_tol=2e-10)
    assert math.isclose(w.Q_total, w.Q_preheat + w.Q_evaporation + w.Q_superheat, rel_tol=2e-12)
    assert math.isclose(w.A_total, w.A_preheat + w.A_evaporation + w.A_superheat, rel_tol=2e-12)
    assert math.isclose(w.UA_total, w.zone_UA_preheat + w.zone_UA_evaporation + w.zone_UA_superheat, rel_tol=2e-12)
    assert math.isclose(result.U_mean, w.UA_total / w.A_total, rel_tol=2e-12)
    assert w.h_out >= w.h_in and w.m_dot_evaporated >= 0.0
    assert w.quality_out is None or 0.0 <= w.quality_out <= 1.0
    assert w.two_phase_pressure_drop_supported is False
print('All energy, area, UA, quality and pressure-drop invariants passed.')

All energy, area, UA, quality and pressure-drop invariants passed.


## Interpretation limits

The model is 0D and supports one active phase-changing side. Two-phase tube pressure drop, CHF, dryout/post-dryout, pure-water evaporation outside tubes, and liquid water carried by a gas (droplets, mist or wall film) are not modelled. Compare `zone_alpha_evaporation`, Q, quality, area, U and UA against trusted calculations; use `inside_alpha_equivalent` only as the whole-exchanger resistance-equivalent diagnostic.